# Airline clean


In [18]:
from pyspark.sql.functions import split, col, trim

df_raw = spark.table("bronze.iata_airlines")

# The column name IS the header string (loader used wrong sep)
# so df_raw.columns[0] == "iata_code^icao_code^name^alias"
raw_col = df_raw.columns[0]

df_iata = (
    df_raw
    .withColumn("parts", split(col(f"`{raw_col}`"), "\\^"))
    .select(
        trim(col("parts")[0]).alias("iata_code"),
        trim(col("parts")[1]).alias("icao_code"),
        trim(col("parts")[2]).alias("airline_name"),
        trim(col("parts")[3]).alias("alias")
    )
    .filter(col("icao_code").isNotNull() & (col("icao_code") != ""))
)

df_iata.show(5)

StatementMeta(, 0fd61d85-54e1-4970-a556-5b099bbb2447, 23, Finished, Available, Finished, False)

+---------+---------+--------------------+-----+
|iata_code|icao_code|        airline_name|alias|
+---------+---------+--------------------+-----+
|       0B|      BMS|            Blue Air|     |
|       0V|      VFC|Vietnam Air Servi...|     |
|       1A|      AGT|             Amadeus|     |
|       1F|      TTF|Infini Travel Inf...|     |
|       1I|      EJA|    Netjets Aviation|     |
+---------+---------+--------------------+-----+
only showing top 5 rows



In [19]:
from pyspark.sql.functions import when, coalesce, lit

df_mapping = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .csv("Files/data/raw/mapping_airlines_csv.csv")
    .select(
        trim(col("icao_code")).alias("icao_code"),
        trim(col("carrier_business_model")).alias("carrier_business_model"),
        trim(col("country")).alias("country")
    )
    .filter(col("icao_code").isNotNull() & (col("icao_code") != ""))
    .dropDuplicates(["icao_code"])
)

df_mapping.groupBy("carrier_business_model").count().orderBy("count", ascending=False).show(10, truncate=False)

StatementMeta(, 0fd61d85-54e1-4970-a556-5b099bbb2447, 24, Finished, Available, Finished, False)

+-----------------------------------+-----+
|carrier_business_model             |count|
+-----------------------------------+-----+
|Full-service / network             |734  |
|Low-cost carrier                   |57   |
|Regional / feeder                  |46   |
|Cargo operator                     |17   |
|Charter / leisure                  |7    |
|Technology / distribution          |5    |
|Business aviation / private charter|2    |
|Government / state operation       |1    |
+-----------------------------------+-----+



In [20]:
display(df_mapping)

StatementMeta(, 0fd61d85-54e1-4970-a556-5b099bbb2447, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0deea61-6290-483e-8663-046643054b23)

In [21]:
df_silver = (
    df_iata
    .join(df_mapping, on="icao_code", how="left")
    .withColumn(
        "carrier_type",
        when(col("carrier_business_model") == "Full-service / network", "Network")
        .when(col("carrier_business_model") == "Low-cost carrier",      "LCC")
        .when(col("carrier_business_model") == "Regional / feeder",     "Regional")
        .otherwise("Other")
    )
    .select("icao_code", "iata_code", "airline_name", "carrier_type", "country")
    .dropDuplicates(["icao_code"])
)

df_silver.groupBy("carrier_type").count().orderBy("count", ascending=False).show()

df_silver.write.mode("overwrite").saveAsTable("silver.airline")
print(f"{df_silver.count()} rows → silver.airline")

StatementMeta(, 0fd61d85-54e1-4970-a556-5b099bbb2447, 26, Finished, Available, Finished, False)

+------------+-----+
|carrier_type|count|
+------------+-----+
|     Network|  734|
|         LCC|   57|
|    Regional|   46|
|       Other|   32|
+------------+-----+

869 rows → silver.airline


In [23]:
display(df_silver)

StatementMeta(, 0fd61d85-54e1-4970-a556-5b099bbb2447, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09140825-eff4-4b3b-8503-3ba8ba9855f1)